Shahlo Solieva

Last updated 07292026

Get atom indices for FR regions and residue numbers for paratope regions. 

In [1]:
import numpy as np
import mdtraj as md
import FR_indices_toolbox
import pandas as pd
import pickle

# Step 1: get framework region atom indices.

NOTE: removed the following residues from the sequence because the reference structure is missing them:

HC FR1: residue 1

LC FR1: residue 1

LC FR4: last residue

In [2]:
BASE_pdb = '/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/clustering/radius_15_variable/data/apo/representative_clusters_07102026/top_cluster_centers/'

In [3]:
# Reference 

partial_seq_list = [
'VQLLESGPGLVRPSETLTLTCSVF',
'YSWIRQPPGRGLEWIAS',
'SRNPSLLSRVTTSIDTERYQVFLNMRSVTAADTAVYFC',
'WGQGTLVVVSS',
'SVLTQPPSASGSPGQSVTISCTGT',
'VSWYQQYPGEAPKVIIY',
'KRPSGVPDRFSGSKSGTTASLTVSGLQAEDEADYYC',
'FGGGTKVTV']

structure_path = '/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/paratope_RMSD/ref_structures/EPTC112_mature_8c8t.pdb'
sequence, ca_indices =FR_indices_toolbox.get_sequence(structure_path)

FR_indices = []
for partial_seq in partial_seq_list:
    partial_seq_ca_indices = FR_indices_toolbox.find_indices(sequence, ca_indices, partial_seq)
    FR_indices_toolbox.partial_seq_check(structure_path, partial_seq_ca_indices, partial_seq)
    FR_indices.extend(partial_seq_ca_indices)

print(f'Number of atoms: {len(FR_indices)}')
np.save(f'../data/EPTC112-ref-FR-indices.npy', FR_indices)

/home/ssolieva/miniconda/envs/pyemma/lib/python3.11/site-packages/mdtraj/formats/pdb/pdbfile.py:200: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn('Unlikely unit cell vectors detected in PDB file likely '


VQLLESGPGLVRPSETLTLTCSVF
YSWIRQPPGRGLEWIAS
SRNPSLLSRVTTSIDTERYQVFLNMRSVTAADTAVYFC
WGQGTLVVVSS
SVLTQPPSASGSPGQSVTISCTGT
VSWYQQYPGEAPKVIIY
KRPSGVPDRFSGSKSGTTASLTVSGLQAEDEADYYC
FGGGTKVTV
Number of atoms: 176


In [4]:
# Mature

sys_name = 'EPTC112_mature_VC'

partial_seq_list = [
'VQLLESGPGLVRPSETLTLTCSVF',
'YSWIRQPPGRGLEWIAS',
'SRNPSLLSRVTTSIDTERYQVFLNMRSVTAADTAVYFC',
'WGQGTLVVVSS',
'SVLTQPPSASGSPGQSVTISCTGT',
'VSWYQQYPGEAPKVIIY',
'KRPSGVPDRFSGSKSGTTASLTVSGLQAEDEADYYC',
'FGGGTKVTV']

structure_path = f'{BASE_pdb}{sys_name}-topcenter.pdb'
sequence, ca_indices =FR_indices_toolbox.get_sequence(structure_path)

FR_indices = []
for partial_seq in partial_seq_list:
    partial_seq_ca_indices = FR_indices_toolbox.find_indices(sequence, ca_indices, partial_seq)
    FR_indices_toolbox.partial_seq_check(structure_path, partial_seq_ca_indices, partial_seq)
    FR_indices.extend(partial_seq_ca_indices)

print(f'Number of atoms: {len(FR_indices)}')
np.save(f'../data/{sys_name}-FR-indices.npy', FR_indices)

VQLLESGPGLVRPSETLTLTCSVF
YSWIRQPPGRGLEWIAS
SRNPSLLSRVTTSIDTERYQVFLNMRSVTAADTAVYFC
WGQGTLVVVSS
SVLTQPPSASGSPGQSVTISCTGT
VSWYQQYPGEAPKVIIY
KRPSGVPDRFSGSKSGTTASLTVSGLQAEDEADYYC
FGGGTKVTV
Number of atoms: 176


In [6]:
# Naive

sys_name = 'EPTC112_naive_VC'

partial_seq_list = [
'VQLQESGPGLVKPSETLSLTCTVS',
'WSWIRQPPGKGLEWIGY',
'NYNPSLKSRVTISVDTSKNQFSLKLSSVTAADTAVYYC',
'WGQGTLVTVSS',
'SALTQPPSASGSPGQSVTISCTGT',
'VSWYQQHPGKAPKLMIY',
'KRPSGVPDRFSGSKSGNTASLTVSGLQAEDEADYYC',
'FGGGTKLTV']

structure_path = f'{BASE_pdb}{sys_name}-topcenter.pdb'
sequence, ca_indices =FR_indices_toolbox.get_sequence(structure_path)

FR_indices = []
for partial_seq in partial_seq_list:
    partial_seq_ca_indices = FR_indices_toolbox.find_indices(sequence, ca_indices, partial_seq)
    FR_indices_toolbox.partial_seq_check(structure_path, partial_seq_ca_indices, partial_seq)
    FR_indices.extend(partial_seq_ca_indices)

print(f'Number of atoms: {len(FR_indices)}')
np.save(f'../data/{sys_name}-FR-indices.npy', FR_indices)

VQLQESGPGLVKPSETLSLTCTVS
WSWIRQPPGKGLEWIGY
NYNPSLKSRVTISVDTSKNQFSLKLSSVTAADTAVYYC
WGQGTLVTVSS
SALTQPPSASGSPGQSVTISCTGT
VSWYQQHPGKAPKLMIY
KRPSGVPDRFSGSKSGNTASLTVSGLQAEDEADYYC
FGGGTKLTV
Number of atoms: 176


# Step 2: get residue numbers for the paratope regions

In [7]:
# Paths
base_path='/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/per_resi_RMSD'
df_base_path =f'{base_path}/data/seq_alignments/naive_aligned'
glycan_base_path='/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/distance_to_antigen/data/to_glycan/'
protein_base_path='/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/distance_to_antigen/data/to_protein/'
BASE_pdb = '/mnt/pure/bowmanlab/WG-shahlo/projects/Ab_dynamics/analysis/clustering/radius_15_variable/data/apo/representative_clusters_07102026/top_cluster_centers/'

def translate_resname_to_rescode(resi_list):
    resname_to_rescode_definition =[[
        'ALA','ARG','ASN','ASP','CYS','GLU','GLN',\
        'GLY','HIS','ILE','LEU','LYS','MET','PHE',\
        'PRO','SER','THR','TRP','TYR','VAL'],\
        ['A'  ,'R'  ,'N'  ,'D'  ,'C'  ,'E'  ,'Q'  ,\
        'G'  ,'H'  ,'I'  ,'L'  ,'K'  ,'M'  ,'F'  ,\
        'P'  ,'S'  ,'T'  ,'W'  ,'Y'  ,'V' ]]
    single_code_resi_list = []
    for i in range(len(resi_list)):
        if len(resi_list[i]) > 1: # if there is a value, then continue.
            resname = str(resi_list[i][0:3])
            res_number = str(resi_list[i][3:])
            for j in range(len(resname_to_rescode_definition[0])):
                if resname == resname_to_rescode_definition[0][j]:
                    single_code_resi_list.append(f'{resname_to_rescode_definition[1][j]}{res_number}')
        if len(resi_list[i]) <= 1: # if there is no value, just append a space.
            single_code_resi_list.append(' ')
    return single_code_resi_list

def plot_per_residue(lineage, which_chain):
    df_glycan    = pd.read_csv(f"{glycan_base_path}{lineage}_{which_chain}.csv")
    df_protein   = pd.read_csv(f"{protein_base_path}{lineage}_{which_chain}.csv")
    glycan_dists = df_glycan[f'pdb_mature_{which_chain}_glycan_minDists']
    protein_dists= df_protein[f'pdb_mature_{which_chain}_protein_minDists']
    
    df = pd.read_csv(f"{df_base_path}/{lineage}_mature_aligned_to_naive_{which_chain}.csv")  
    mature_resi=  df[f'mature_{which_chain}_resi']

    print(len(protein_dists), len(glycan_dists), len(mature_resi))
    paratope_resis= []
    paratope_names= []
    paratope_pymol= []
    for i in range(len(protein_dists)):
        if protein_dists[i] < 10 or glycan_dists[i] < 10:
            paratope_names.append(mature_resi[i])
            paratope_pymol.append(str(i+1))
            paratope_resis.append(str(i))

    return paratope_resis, paratope_names, paratope_pymol

In [8]:
lineage='EPTC112' 
sys_name = 'EPTC112_mature_VC'
structure = md.load(f'{BASE_pdb}{sys_name}-topcenter.pdb')

In [10]:
paratope_resis, paratope_names, paratope_pymol = plot_per_residue(lineage, 'HC')
resid_selection_str = ' '.join(paratope_pymol)
print('HC paratope resis:',resid_selection_str)
print(paratope_names)

121 121 121
HC paratope resis: 2 3 19 20 21 24 25 26 27 28 29 30 31 32 33 34 52 53 54 55 56 68 69 70 71 72 73 74 75 76 77 78 79 80 81 97 98 99 100 101 102 103 104 105 106 110
['VAL2', 'GLN3', 'THR19', 'LEU20', 'THR21', 'VAL24', 'PHE25', 'ASN26', 'SER27', 'ARG28', 'VAL29', 'SER30', 'GLY31', 'TYR32', 'TYR33', 'TYR34', 'HIS52', 'PHE53', 'SER54', 'LEU55', 'ARG56', 'THR68', 'THR69', 'SER70', 'ILE71', 'ASP72', 'THR73', 'GLU74', 'ARG75', 'TYR76', 'GLN77', 'VAL78', 'PHE79', 'LEU80', 'ASN81', 'ARG97', 'GLY98', 'ASP99', 'ALA100', 'SER101', 'GLY102', 'TRP103', 'ARG104', 'ALA105', 'ASP106', 'HIS110']


In [11]:
paratope_resis, paratope_names, paratope_pymol = plot_per_residue(lineage, 'LC')
resid_selection_str = ' '.join(paratope_pymol)
print('LC paratope resis:',resid_selection_str)
print(paratope_names)

110 110 110
LC paratope resis: 32 34 97
['SER32', 'TYR34', 'HIS97']
